In [2]:
# =============================================================================
# Cell 1: Load packages
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(arrow)       # read parquet files
  library(ggplot2)
  library(pheatmap)
  library(dplyr)
  library(tibble)
  library(tidyr)
  library(readr)
})
message("All packages loaded")

# =============================================================================
# Cell 2: Configuration 
# =============================================================================
BASE <- "/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks"

# ⬇️ UPDATED PATH: Pointing to the new fragment matrices folder
QUANT_DIR <- file.path(BASE, "cross_species_consensus_v3/12_fragment_matrices")

# We'll save the DESeq2 outputs to a new folder so we don't overwrite old stuff
OUT_DIR <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

SPECIES <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")


All packages loaded



In [3]:
# =============================================================================
# Cell 3 & 4: Load and merge new Pseudobulk data & metadata
# =============================================================================
all_counts <- list()
all_meta <- list()

for (sp in SPECIES) {
  sp_dir <- file.path(QUANT_DIR, sp)
  counts_file <- file.path(sp_dir, "pseudobulk_counts.parquet")
  meta_file <- file.path(sp_dir, "pseudobulk_groups.parquet") 
  
  if (!file.exists(counts_file)) {
    message("Skipping ", sp, ": files not found.")
    next
  }
  
  # Load counts
  counts <- as.data.frame(read_parquet(counts_file))
  
  # ⬇️ THE NEW INSPECT & FAST-MERGE LOGIC
  if (any(duplicated(counts$region_id))) {
    dup_ids <- unique(counts$region_id[duplicated(counts$region_id)])
    message("\n  ⚠️ Found ", length(dup_ids), " duplicated region IDs in ", sp, "!")
    
    # 1. INSPECT: Isolate all rows that share these duplicated IDs
    duplicate_df <- counts[counts$region_id %in% dup_ids, ]
    duplicate_df <- duplicate_df[order(duplicate_df$region_id), ] # Sort so duplicates are next to each other
    
    message("  🔍 Inspecting the first few columns of the duplicates:")
    # Print just the region_id and the first 3 numeric columns so it doesn't flood your screen
    print(duplicate_df[, 1:min(4, ncol(duplicate_df))])
    
    # Optional: Save them to a CSV so you can look at the full data in Excel/VSCode
    inspect_file <- file.path(sp_dir, paste0("duplicated_regions_inspect.csv"))
    write.csv(duplicate_df, inspect_file, row.names = FALSE)
    message("  💾 Saved full duplicate dataframe to: ", inspect_file)
    
    # 2. FAST MERGE: Use base R's C-optimized rowsum instead of dplyr
    message("  ⚡ Fast-merging duplicates...")
    numeric_mat <- as.matrix(counts[, -which(names(counts) == "region_id")])
    merged_mat <- rowsum(numeric_mat, group = counts$region_id)
    
    # Rebuild the dataframe
    counts <- as.data.frame(merged_mat)
    counts$region_id <- rownames(counts)
  } else {
    rownames(counts) <- counts$region_id
  }
  
  counts$region_id <- NULL
  
  # Load metadata 
  meta <- as.data.frame(read_parquet(meta_file))
  
  # Prefix the sample labels with the species
  new_labels <- paste0(sp, "_", meta$label)
  colnames(counts) <- new_labels
  
  meta$sample_id <- new_labels
  meta$species <- sp
  
  all_counts[[sp]] <- counts
  all_meta[[sp]] <- meta
  message("Loaded ", sp, ": ", ncol(counts), " pseudobulk samples")
}

# 1. Inner join counts on shared peaks
shared_peaks <- Reduce(intersect, lapply(all_counts, rownames))
counts_merged <- do.call(cbind, unname(lapply(all_counts, function(x) x[shared_peaks, , drop = FALSE])))
counts_merged[is.na(counts_merged)] <- 0

# 2. Bind metadata 
meta_merged <- do.call(rbind, unname(all_meta))
rownames(meta_merged) <- meta_merged$sample_id

# 3. Ensure columns and rows align perfectly
counts_merged <- counts_merged[, rownames(meta_merged)]

message("\nMerged matrix: ", nrow(counts_merged), " peaks x ", ncol(counts_merged), " samples (donor pseudobulks)")

Loaded Human: 165 pseudobulk samples


  ⚠️ Found 27 duplicated region IDs in Bonobo!

  🔍 Inspecting the first few columns of the duplicates:



                           region_id Adipocytes__BN114 Adipocytes__BN116
50013          1:121016937-121017437                 0                 0
497563         1:121016937-121017437                 0                 0
834                1:1584884-1585384                 0                 0
892                1:1584884-1585384                 0                 2
146538            11:3407913-3408413                 0                 0
169060            11:3407913-3408413                 0                 0
216454          12:31363611-31364111                 0                 0
963706          12:31363611-31364111                 0                 6
298621        14:106929440-106929940                 0                 0
298732        14:106929440-106929940                 2                 0
268948          14:22951559-22952059                 0                 0
268968          14:22951559-22952059                 0                 0
340790          16:15714719-15715219               

  💾 Saved full duplicate dataframe to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Bonobo/duplicated_regions_inspect.csv

  ⚡ Fast-merging duplicates...

Loaded Bonobo: 102 pseudobulk samples


  ⚠️ Found 14 duplicated region IDs in Chimpanzee!

  🔍 Inspecting the first few columns of the duplicates:



                         region_id Adipocytes__BN075 Adipocytes__BN085
54204        1:112237532-112238032                 0                 0
55689        1:112237532-112238032                 0                 6
77712        1:181505502-181506002                 0                 0
311705       1:181505502-181506002                 0                 2
940                1:873918-874418                 0                 0
1007               1:873918-874418                 0                 2
178225        11:71575021-71575521                 0                 0
913952        11:71575021-71575521                 0                 0
278301          14:7730246-7730746                 0                 0
278328          14:7730246-7730746                 0                 0
308891        14:91678084-91678584                 0                 0
309016        14:91678084-91678584                 0                 0
352809        16:16575631-16576131                 0                 0
354027

  💾 Saved full duplicate dataframe to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Chimpanzee/duplicated_regions_inspect.csv

  ⚡ Fast-merging duplicates...

Loaded Chimpanzee: 103 pseudobulk samples


  ⚠️ Found 30 duplicated region IDs in Gorilla!

  🔍 Inspecting the first few columns of the duplicates:



...

  💾 Saved full duplicate dataframe to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Gorilla/duplicated_regions_inspect.csv

  ⚡ Fast-merging duplicates...

Loaded Gorilla: 161 pseudobulk samples


  ⚠️ Found 45 duplicated region IDs in Macaque!

  🔍 Inspecting the first few columns of the duplicates:



                    region_id Adipocytes__Becky_2616 Adipocytes__Nazisse_2551
51400   1:103938687-103939187                      0                        0
52818   1:103938687-103939187                      0                        2
49913   1:107662425-107662925                      0                        0
489663  1:107662425-107662925                      0                        0
10024   1:207816002-207816502                      0                        0
10158   1:207816002-207816502                      0                        0
872     1:222731073-222731573                      0                        0
929     1:222731073-222731573                      0                        0
411139   10:40106469-40106969                      0                        0
565238   10:40106469-40106969                      0                        0
537525 12:129505382-129505882                      0                        0
705128 12:129505382-129505882                      0            

  💾 Saved full duplicate dataframe to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Macaque/duplicated_regions_inspect.csv

  ⚡ Fast-merging duplicates...

Loaded Macaque: 60 pseudobulk samples


  ⚠️ Found 77 duplicated region IDs in Marmoset!

  🔍 Inspecting the first few columns of the duplicates:



                             region_id Adipocytes__Almerida_17079
886674           1:159656579-159657079                          0
886679           1:159656579-159657079                          0
553340           1:191365166-191365666                          0
553345           1:191365166-191365666                          0
474202           1:216823722-216824222                          0
569229           1:216823722-216824222                          6
238422             1:62211874-62212374                          0
238489             1:62211874-62212374                          0
281882            10:22690391-22690891                          0
281894            10:22690391-22690891                          0
286565            10:28337912-28338412                          0
286665            10:28337912-28338412                          0
152438          11:113040682-113041182                          0
152464          11:113040682-113041182                          0
161032    

  💾 Saved full duplicate dataframe to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/12_fragment_matrices/Marmoset/duplicated_regions_inspect.csv

  ⚡ Fast-merging duplicates...

Loaded Marmoset: 87 pseudobulk samples


Merged matrix: 0 peaks x 678 samples (donor pseudobulks)



In [3]:


# =============================================================================
# Cell 5: Setup Experimental Design Metadata
# =============================================================================
# Convert string columns to factors for DESeq2
meta_merged$species   <- factor(meta_merged$species, levels = SPECIES)
meta_merged$cell_type <- factor(meta_merged$cell_type)
meta_merged$region    <- factor(meta_merged$region)

# Create the specific contrast variable for Human vs Everyone else
meta_merged$is_human <- ifelse(meta_merged$species == "Human", "Human", "NonHuman")

# Set NonHuman as the reference level, so positive log2FoldChange = Higher in Human
meta_merged$is_human <- factor(meta_merged$is_human, levels = c("NonHuman", "Human"))

# Find cell types that are shared across species (optional, but good for clean models)
ct_per_species <- split(as.character(meta_merged$cell_type), meta_merged$species)
shared_ct <- Reduce(intersect, ct_per_species)

# If you want to use ALL cell types, you can comment out these next 4 lines.
keep_samples <- meta_merged$cell_type %in% shared_ct
counts_shared <- counts_merged[, keep_samples]
meta_shared   <- meta_merged[keep_samples, ]
meta_shared$cell_type <- droplevels(meta_shared$cell_type)

message("Subset to shared cell types: ", ncol(counts_shared), " pseudobulk samples")
message("Cell types: ", paste(levels(meta_shared$cell_type), collapse = ", "))


Subset to shared cell types: 411 pseudobulk samples

Cell types: Colonocytes, Crypt Fibroblasts WNT2B+, ECs, EECs, Enteric glia, Enterocytes, Goblet cells, Lymphatic ECs, Macrophages, Myofibroblasts, Naive B cells, Plasma B cells, Specialized Fibroblasts KCNN3+, Specialized Fibroblasts RSPO3+ only, Specialized Fibroblasts SYNM+, Stem cells, T cells, TA cells



In [4]:
# =============================================================================
# Cell 6: Global VST and PCA (Quality Control)
# =============================================================================
# We run a quick global DESeq2 just to get the Variance Stabilized Transform (VST).
# This is required for PCA and Heatmaps so the data is log-scaled and normalized.

message("Preparing global data for PCA...")
# Light filtering for the global PCA
keep_global <- rowSums(counts_shared >= 10) >= 10
dds_global <- DESeqDataSetFromMatrix(
  countData = round(counts_shared[keep_global, ]),
  colData   = meta_shared,
  design    = ~ cell_type + is_human
)
dds_global <- estimateSizeFactors(dds_global)
vsd_global <- vst(dds_global, blind = FALSE)



Preparing global data for PCA...

converting counts to integer mode

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

using ntop=500 top features by variance



ERROR: [1m[33mError[39m in `ggsave()`:[22m
[1m[22m[33m![39m Cannot find directory
  [34m/cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v2/13_deseq2_R_pseudobulk/plots[39m.
[36mℹ[39m Please supply an existing directory or use `create.dir = TRUE`.


In [5]:
# 1. Plot Global PCA
pca_data <- plotPCA(vsd_global, intgroup = c("cell_type", "is_human"), returnData = TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

pca_plot <- ggplot(pca_data, aes(x = PC1, y = PC2, color = cell_type, shape = is_human)) +
  geom_point(size = 3, alpha = 0.8) +
  xlab(paste0("PC1: ", percentVar[1], "% variance")) +
  ylab(paste0("PC2: ", percentVar[2], "% variance")) +
  theme_bw() +
  ggtitle("Global PCA of Pseudobulks")

# Save PCA
ggsave(file.path(OUT_DIR, "plots", "Global_PCA.pdf"), pca_plot, width = 8, height = 6)
message("✅ Global PCA saved to plots directory.")



using ntop=500 top features by variance

✅ Global PCA saved to plots directory.



In [6]:
# =============================================================================
# Cell 6: Global VST and Custom PCA
# =============================================================================
message("Preparing global data for PCA...")
# Light filtering for the global PCA
keep_global <- rowSums(counts_shared >= 10) >= 10
dds_global <- DESeqDataSetFromMatrix(
  countData = round(counts_shared[keep_global, ]),
  colData   = meta_shared,
  design    = ~ cell_type + is_human
)
dds_global <- estimateSizeFactors(dds_global)
vsd_global <- vst(dds_global, blind = FALSE)

# 1. Extract raw PCA data with all the metadata we need
pca_data <- plotPCA(vsd_global, intgroup = c("cell_type", "species", "donor"), returnData = TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

# 2. Filter and collapse cell types
target_cts <- c("Enterocytes", "Colonocytes", "Goblet cells", "T cells", "Naive B cells", "Plasma B cells")

pca_filtered <- pca_data %>%
  mutate(cell_type_broad = case_when(
    cell_type %in% target_cts ~ as.character(cell_type),
    grepl("Fibroblast", cell_type) ~ "Fibroblasts", # Collapse all fibroblast subtypes
    TRUE ~ "Drop"
  )) %>%
  filter(cell_type_broad != "Drop")

pca_filtered$cell_type_broad <- factor(pca_filtered$cell_type_broad)

# 3. Build the custom plot
# We use color for species, shape for cell type, and ggrepel to add donor text labels
pca_plot <- ggplot(pca_filtered, aes(x = PC1, y = PC2, color = species, shape = cell_type_broad)) +
  geom_point(size = 4, alpha = 0.85) +
  ggrepel::geom_text_repel(aes(label = donor), size = 3, show.legend = FALSE, max.overlaps = 20) +
  
  # Manually set 7 shapes since ggplot normally maxes out at 6
  scale_shape_manual(values = c(15, 16, 17, 18, 3, 4, 8)) + 
  
  xlab(paste0("PC1: ", percentVar[1], "% variance")) +
  ylab(paste0("PC2: ", percentVar[2], "% variance")) +
  theme_bw() +
  ggtitle("PCA: Selected Cell Types (Colored by Species, Labeled by Donor)") +
  theme(
    legend.position = "right",
    legend.title = element_text(face = "bold"),
    plot.title = element_text(face = "bold", size = 14)
  )

# Save the plot
ggsave(file.path(OUT_DIR, "plots", "Custom_Filtered_PCA.pdf"), pca_plot, width = 10, height = 7)
message("✅ Custom PCA saved to plots directory.")

Preparing global data for PCA...

converting counts to integer mode

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

using ntop=500 top features by variance

Warning message:
“ggrepel: 210 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
✅ Custom PCA saved to plots directory.



In [8]:
# =============================================================================
# Cell 6: Species-Aware Peak Filtering
# =============================================================================
MIN_SAMPLES_PER_SPECIES <- 2  # How many samples within a species need signal?
MIN_SPECIES_ACTIVE <- 6      # How many species must pass the above threshold?

n_before <- nrow(counts_shared)

# 1. Filter: Max count across any sample > 150 (removes total background noise)
keep_count <- apply(counts_shared, 1, max) >= 150

# 2. Filter: Species-aware presence
# Create a logical matrix: Peaks (rows) x Species (cols)
active_in_species <- sapply(SPECIES, function(sp) {
  sp_cols <- meta_shared$species == sp
  if (sum(sp_cols) > 0) {
    # TRUE if at least MIN_SAMPLES_PER_SPECIES have >= 10 fragments
    rowSums(counts_shared[, sp_cols, drop = FALSE] >= 10) >= MIN_SAMPLES_PER_SPECIES
  } else {
    rep(FALSE, nrow(counts_shared))
  }
})

# A peak is kept if it is active in at least MIN_SPECIES_ACTIVE
keep_species_thresh <- rowSums(active_in_species) >= MIN_SPECIES_ACTIVE

keep_peaks <- keep_count & keep_species_thresh
counts_shared <- counts_shared[keep_peaks, ]

message("Species-Aware Peak filtering:")
message("  Kept: ", sum(keep_peaks), " (", round(100 * sum(keep_peaks) / n_before, 1), "%)")

# =============================================================================
# Cell 6b: Global VST and Custom PCA (Post-Filtering)
# =============================================================================
message("\nPreparing global data for PCA based on species-aware filtered peaks...")

# Run a global DESeq2 model purely to get the normalized, log-scaled data (VST)
dds_global <- DESeqDataSetFromMatrix(
  countData = round(counts_shared),
  colData   = meta_shared,
  design    = ~ species + cell_type
)

# Estimate size factors and apply Variance Stabilizing Transformation (VST)
dds_global <- estimateSizeFactors(dds_global)
vsd_global <- vst(dds_global, blind = FALSE)

message("Extracting PCA coordinates...")

# 1. Extract raw PCA data 
pca_data <- plotPCA(vsd_global, intgroup = c("cell_type", "species", "donor"), returnData = TRUE)
percentVar <- round(100 * attr(pca_data, "percentVar"))

# 2. Filter and collapse cell types for visualization
target_cts <- c("Enterocytes", "Colonocytes", "Goblet cells", "T cells", "Naive B cells", "Plasma B cells")

pca_filtered <- pca_data %>%
  mutate(cell_type_broad = case_when(
    cell_type %in% target_cts ~ as.character(cell_type),
    grepl("Fibroblast", cell_type) ~ "Fibroblasts", # Collapse all fibroblast subtypes
    TRUE ~ "Drop"
  )) %>%
  filter(cell_type_broad != "Drop")

pca_filtered$cell_type_broad <- factor(pca_filtered$cell_type_broad)

message(sprintf("Plotting PCA for %d selected samples...", nrow(pca_filtered)))

# 3. Build the custom plot
# - Color: Species
# - Shape: Cell Type
# - Label: Donor
pca_plot <- ggplot(pca_filtered, aes(x = PC1, y = PC2, color = species, shape = cell_type_broad)) +
  geom_point(size = 4, alpha = 0.85) +
  ggrepel::geom_text_repel(aes(label = donor), size = 3, show.legend = FALSE, max.overlaps = 20) +
  
  # Manually set 7 distinct shapes
  scale_shape_manual(values = c(15, 16, 17, 18, 3, 4, 8)) + 
  
  xlab(paste0("PC1: ", percentVar[1], "% variance")) +
  ylab(paste0("PC2: ", percentVar[2], "% variance")) +
  theme_bw() +
  ggtitle("Global PCA (Conserved Peaks)") +
  theme(
    legend.position = "right",
    legend.title = element_text(face = "bold"),
    plot.title = element_text(face = "bold", size = 14)
  )

# Save the plot
ggsave(file.path(OUT_DIR, "plots", "Custom_Filtered_PCA_2.pdf"), pca_plot, width = 10, height = 7)
message("✅ Custom PCA saved to: ", file.path(OUT_DIR, "plots", "Custom_Filtered_PCA_2.pdf"))

Species-Aware Peak filtering:

  Kept: 123101 (70.1%)


Preparing global data for PCA based on species-aware filtered peaks...

converting counts to integer mode

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

Extracting PCA coordinates...

using ntop=500 top features by variance

Plotting PCA for 230 selected samples...

Warning message:
“ggrepel: 46 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
✅ Custom PCA s

ERROR: Error in checkContrast(contrast, resNames): 'contrast', as a list of length 2, should have character vectors as elements,
see the manual page of ?results for more information


In [9]:
# =============================================================================
# Cell 7: Per-Cell-Type DESeq2 (Testing ALL Species)
# =============================================================================
res_list <- list() 

cell_types <- levels(meta_shared$cell_type)

for (ct in cell_types) {
  message(sprintf("\n=== Processing %s ===", ct))
  
  meta_ct <- meta_shared[meta_shared$cell_type == ct, ]
  counts_ct <- counts_shared[, rownames(meta_ct)]
  
  # Ensure the cell type is actually present in multiple species
  present_species <- unique(as.character(meta_ct$species))
  if (length(present_species) < 2) {
    message("  ⚠️ Skipping: Not enough species represented for this cell type.")
    next
  }
  
  # Local peak filtering for this specific cell type
  keep_ct <- rowSums(counts_ct >= 10) >= 3
  counts_ct_filtered <- counts_ct[keep_ct, ]
  
  # ⬇️ THE FIX 1: Use ~ 0 + species (removes intercept for clean contrasts)
  dds_ct <- DESeqDataSetFromMatrix(
    countData = round(counts_ct_filtered),
    colData   = meta_ct,
    design    = ~ 0 + species
  )
  
  dds_ct <- DESeq(dds_ct, quiet = TRUE)
  
  res_list[[ct]] <- list()
  
  # Get the exact term names (e.g., "speciesHuman", "speciesMacaque")
  res_names <- resultsNames(dds_ct)
  
  for (target_sp in present_species) {
    
    target_term <- paste0("species", target_sp)
    if (!target_term %in% res_names) next
    
    # Build a numeric contrast vector: Target species = 1, All others = -1 / (N-1)
    contrast_vec <- rep(-1 / (length(res_names) - 1), length(res_names))
    target_idx <- which(res_names == target_term)
    contrast_vec[target_idx] <- 1
    
    # ⬇️ THE FIX 2: Pass the numeric vector directly, not inside a list()
    res_sp <- results(dds_ct, contrast = contrast_vec)
    res_sp_ordered <- res_sp[order(res_sp$padj), ]
    
    # Save it to our nested list 
    res_list[[ct]][[target_sp]] <- res_sp_ordered
    
    sig_hits <- sum(res_sp$padj < 0.05 & res_sp$log2FoldChange > 1, na.rm = TRUE)
    message(sprintf("  ✅ %-12s specific regions: %d (padj < 0.05, LFC > 1)", target_sp, sig_hits))
  }
}


=== Processing Colonocytes ===

converting counts to integer mode

  ✅ Human        specific regions: 11184 (padj < 0.05, LFC > 1)

  ✅ Bonobo       specific regions: 851 (padj < 0.05, LFC > 1)

  ✅ Chimpanzee   specific regions: 567 (padj < 0.05, LFC > 1)

  ✅ Gorilla      specific regions: 4144 (padj < 0.05, LFC > 1)

  ✅ Macaque      specific regions: 5925 (padj < 0.05, LFC > 1)

  ✅ Marmoset     specific regions: 9903 (padj < 0.05, LFC > 1)


=== Processing Crypt Fibroblasts WNT2B+ ===

converting counts to integer mode

  ✅ Human        specific regions: 5512 (padj < 0.05, LFC > 1)

  ✅ Bonobo       specific regions: 5766 (padj < 0.05, LFC > 1)

  ✅ Chimpanzee   specific regions: 1440 (padj < 0.05, LFC > 1)

  ✅ Gorilla      specific regions: 3124 (padj < 0.05, LFC > 1)

  ✅ Macaque      specific regions: 2149 (padj < 0.05, LFC > 1)

  ✅ Marmoset     specific regions: 7078 (padj < 0.05, LFC > 1)


=== Processing ECs ===

converting counts to integer mode

  ✅ Human        specifi

In [10]:
# =============================================================================
# Cell 8: Top Species-Specific Heatmap (Enterocytes)
# =============================================================================
target_ct <- "Enterocytes"
message("\n=== Generating Species-Specific Heatmap for: ", target_ct, " ===")

top_peaks_list <- list()

# Extract top 25 significant regions per species (padj < 0.05, LFC > 1)
for (sp in names(res_list[[target_ct]])) {
  res_sp <- res_list[[target_ct]][[sp]]
  
  # Filter for significance and positive LFC (means higher in THIS species)
  sig_res <- res_sp[!is.na(res_sp$padj) & res_sp$padj < 0.05 & res_sp$log2FoldChange > 1, ]
  
  # Sort by fold change to get the strongest biological markers
  sig_res <- sig_res[order(sig_res$log2FoldChange, decreasing = TRUE), ] 
  top_peaks_list[[sp]] <- rownames(head(sig_res, 25))
}

# Combine and get unique peaks (max 150 regions if 6 species * 25)
top_regions <- unique(unlist(top_peaks_list))

# Subset metadata and VST data for Enterocytes
meta_ct <- meta_shared[meta_shared$cell_type == target_ct, ]

# Sort metadata by species so the heatmap columns form neat species blocks
meta_ct <- meta_ct[order(meta_ct$species), ]
samples_to_plot <- rownames(meta_ct)

# Get log-normalized values from the global VST object we made earlier
mat <- assay(vsd_global)[top_regions, samples_to_plot]

# Row scaling (Z-score) so high/low peaks are visually comparable
mat_scaled <- t(scale(t(mat)))

# Create Annotation dataframe
anno_col <- data.frame(
  Species = meta_ct$species,
  row.names = samples_to_plot
)

# Draw Heatmap
heatmap_file <- file.path(OUT_DIR, "plots", paste0("Heatmap_", target_ct, "_Species_Markers.pdf"))

pheatmap(mat_scaled,
         annotation_col = anno_col,
         cluster_cols = FALSE,  # 👈 FALSE keeps the species neatly grouped together!
         cluster_rows = TRUE,   # Groups similar peaks together
         show_rownames = FALSE,
         show_colnames = FALSE,
         main = paste("Top Species-Specific Regions in", target_ct),
         filename = heatmap_file,
         width = 9, height = 7)

message("✅ Heatmap saved to: ", heatmap_file)




=== Generating Species-Specific Heatmap for: Enterocytes ===

✅ Heatmap saved to: /cluster/project/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v2/13_deseq2_R_pseudobulk/plots/Heatmap_Enterocytes_Species_Markers.pdf



In [ ]:
# =============================================================================
# Cell 9: Human-Specific Regions - Cell Type Sharing Analysis
# =============================================================================
message("\n=== Analyzing Human-Specific Region Sharing Across Cell Types ===")

target_species <- "Human"
human_hits_per_ct <- list()

# Collect all Human-specific hits from every cell type
for (ct in names(res_list)) {
  if (target_species %in% names(res_list[[ct]])) {
    res <- res_list[[ct]][[target_species]]
    
    # Filter for significant Human hits
    sig_peaks <- rownames(res[!is.na(res$padj) & res$padj < 0.05 & res$log2FoldChange > 1, ])
    if (length(sig_peaks) > 0) {
      human_hits_per_ct[[ct]] <- sig_peaks
    }
  }
}

# Master list of all unique human peaks
all_human_peaks <- unique(unlist(human_hits_per_ct))
message("Total unique human-specific peaks across all cell types: ", length(all_human_peaks))

# Create a logical presence/absence matrix (Peaks x CellTypes)
overlap_mat <- sapply(human_hits_per_ct, function(peaks) {
  all_human_peaks %in% peaks
})
rownames(overlap_mat) <- all_human_peaks

# Count how many cell types each peak is active in
peak_ct_counts <- rowSums(overlap_mat)

# Print Summary
sharing_summary <- table(peak_ct_counts)
message("\nDistribution of Human Peak Sharing:")
for (i in seq_along(sharing_summary)) {
  message(sprintf("  Active in exactly %s cell type(s): %d peaks", names(sharing_summary)[i], sharing_summary[i]))
}

# Plot the distribution
sharing_df <- data.frame(CellTypesCount = as.numeric(names(sharing_summary)),
                         NumPeaks = as.numeric(sharing_summary))

bar_plot <- ggplot(sharing_df, aes(x = factor(CellTypesCount), y = NumPeaks)) +
  geom_bar(stat = "identity", fill = "indianred") +
  geom_text(aes(label = NumPeaks), vjust = -0.5, size = 3.5) +
  theme_classic() +
  labs(title = "Human-Specific Regions: Sharing Across Cell Types",
       x = "Number of Cell Types", y = "Number of Regions") +
  theme(plot.title = element_text(face = "bold"))

ggsave(file.path(OUT_DIR, "plots", "Human_Peak_Sharing_Summary.pdf"), bar_plot, width = 7, height = 5)
message("✅ Sharing summary barplot saved to plots directory.")

In [ ]:
# =============================================================================
# Cell 7: Per-Cell-Type DESeq2 Loop
# =============================================================================
# We will store the DESeq2 results for each cell type in this list
res_list <- list()

cell_types <- levels(meta_shared$cell_type)

for (ct in cell_types) {
  message(sprintf("\n=== Processing %s ===", ct))
  
  # 1. Subset metadata and counts
  meta_ct <- meta_shared[meta_shared$cell_type == ct, ]
  counts_ct <- counts_shared[, rownames(meta_ct)]
  
  # Ensure we have both Human and NonHuman samples to compare
  if (length(unique(meta_ct$is_human)) < 2) {
    message("  ⚠️ Skipping: Needs both Human and NonHuman samples.")
    next
  }
  
  # 2. Filter peaks SPECIFICALLY for this cell type
  # Keep peaks that have at least 10 counts in at least 3 samples of this cell type
  keep_ct <- rowSums(counts_ct >= 10) >= 3
  counts_ct_filtered <- counts_ct[keep_ct, ]
  
  message("  Testing ", nrow(counts_ct_filtered), " active peaks across ", ncol(counts_ct_filtered), " samples.")
  
  # 3. Run DESeq2
  # NOTE: Using ~ is_human. If you want to control for region here, you can try 
  # ~ region + is_human, but it will fail if a region lacks Human/NonHuman balance.
  dds_ct <- DESeqDataSetFromMatrix(
    countData = round(counts_ct_filtered),
    colData   = meta_ct,
    design    = ~ is_human
  )
  
  dds_ct <- DESeq(dds_ct, quiet = TRUE)
  res_ct <- results(dds_ct, contrast = c("is_human", "Human", "NonHuman"))
  
  # Order by significance and save
  res_ct_ordered <- res_ct[order(res_ct$padj), ]
  res_list[[ct]] <- res_ct_ordered
  
  # Print quick summary
  sig_hits <- sum(res_ct$padj < 0.05 & abs(res_ct$log2FoldChange) > 1, na.rm = TRUE)
  message("  ✅ Found ", sig_hits, " significant human-specific regions (padj < 0.05, |LFC| > 1).")
}

# =============================================================================
# Cell 8: Visualize Top Hits in a Heatmap
# =============================================================================
# Let's pick a specific cell type to visualize (e.g., the first one in the list)
target_ct <- cell_types[1] # You can change this to "Enterocytes", "Stem_cells", etc.
message("\nGenerating heatmap for top regions in: ", target_ct)

# Get the top 50 significant regions for this cell type
res_target <- res_list[[target_ct]]
top_regions <- rownames(head(res_target[which(res_target$padj < 0.05), ], 50))

if (length(top_regions) < 2) {
  message("  ⚠️ Not enough significant regions to draw a heatmap.")
} else {
  # Subset the GLOBAL variance-stabilized data for just these peaks and samples
  samples_to_plot <- rownames(meta_shared[meta_shared$cell_type == target_ct, ])
  mat <- assay(vsd_global)[top_regions, samples_to_plot]
  
  # Scale the data by row (Z-score) so high/low peaks are comparable visually
  mat_scaled <- t(scale(t(mat)))
  
  # Set up the annotation bar at the top of the heatmap
  anno_col <- data.frame(
    Species = meta_shared[samples_to_plot, "species"],
    Region = meta_shared[samples_to_plot, "region"],
    row.names = samples_to_plot
  )
  
  # Plot and save
  heatmap_file <- file.path(OUT_DIR, "plots", paste0("Heatmap_", target_ct, "_Top50.pdf"))
  
  pheatmap(mat_scaled, 
           annotation_col = anno_col,
           cluster_rows = TRUE, 
           cluster_cols = TRUE, 
           show_rownames = FALSE, # Set TRUE if you want to see the unified_ IDs, but 50 gets crowded
           show_colnames = FALSE,
           main = paste("Top 50 Human-Specific Regions in", target_ct),
           filename = heatmap_file,
           width = 8, height = 6)
  
  message("✅ Heatmap saved to: ", heatmap_file)
}